# PDF Table Inspection

ACTIONS 표의 AND/OR indent 좌표 테스트. `pdfplumber.extract_words()`로 단어 좌표를 뽑아 들여쓰기 기준 클러스터링이 되는지 확인.

In [ ]:
import re
import json
import pdfplumber
from collections import OrderedDict, defaultdict

# ══════════════════════════════════════════════════════════
# 설정
# ══════════════════════════════════════════════════════════
PDF_PATH   = "../data/raw/vol1/NUREG-1431-Vol.1-Rev.5-LCO.pdf"
START_PAGE = 243          # 1-indexed, 포함
END_PAGE   = 247          # 1-indexed, 포함
OUT_DIR    = "../data/processed"
SOURCE_DOC = "NUREG-1431 Vol.1"

COL1_MAX = 220
COL2_MAX = 410
HEADER_MARGIN = 95

# ══════════════════════════════════════════════════════════
# 정규식
# ══════════════════════════════════════════════════════════
RE_ACTION_LABEL = re.compile(r'^[A-Z]\.\d+(?:\.\d+)?$')
RE_CONNECTOR    = re.compile(r'^(?:AND|OR)$')
RE_COND_LABEL   = re.compile(r'^[A-Z]\.$')
RE_FOOTER_LEFT  = re.compile(r'Westinghouse\s+STS|BWR/\d|CE\b')
RE_FOOTER_LCO   = re.compile(r'^(\d+\.\d+\.\d+)-\d+$')
RE_FOOTER_REV   = re.compile(r'^Rev\.')
RE_APPLIC       = re.compile(r'APPLICABILITY', re.IGNORECASE)
RE_ACTIONS_HDR  = re.compile(r'\bACTIONS\b')
RE_LCO_HEADER   = re.compile(r'\b(\d\.\d+\.\d+)\b')
RE_STRIP_ALABEL = re.compile(r'^[A-Z]\.\d+(?:\.\d+)?\s*')
RE_STRIP_CLABEL = re.compile(r'^[A-Z]\.\s*')
RE_SR_REF       = re.compile(r'SR\s+\d+\.\d+\.\d+\.\d+')


# ══════════════════════════════════════════════════════════
# 정제
# ══════════════════════════════════════════════════════════
def clean_text(txt):
    txt = re.sub(r"-{3,}\s*REVIEWER['’]S\s+NOTE.*?-{3,}", "", txt,
                 flags=re.DOTALL | re.IGNORECASE)
    txt = re.sub(r"-{3,}\s*NOTE\s*-{3,}(.*?)-{3,}",
                 lambda m: f"(NOTE: {re.sub(r'\\s+', ' ', m.group(1)).strip()})",
                 txt, flags=re.DOTALL)
    txt = txt.replace('[', '').replace(']', '')
    return re.sub(r'\s+', ' ', txt).strip()

def extract_note(txt):
    """운영 NOTE 본문만 추출 (없으면 None). clean_text 적용된 텍스트 기준."""
    m = re.search(r'\(NOTE:\s*(.*?)\)', txt)
    return m.group(1).strip() if m else None

def strip_note(txt):
    return re.sub(r'\(NOTE:.*?\)\s*', '', txt).strip()

def strip_bracket(t):
    return t.lstrip('[').strip()

def is_optional(raw):
    return ('[' in raw) or (']' in raw)


# ══════════════════════════════════════════════════════════
# 라벨 계층
# ══════════════════════════════════════════════════════════
def parse_label(label):
    m = re.match(r'^([A-Z])\.(\d+)(?:\.(\d+))?$', label)
    if not m:
        return None
    return {"condition": m.group(1),
            "group": int(m.group(2)) if m.group(2) else None,
            "alt": int(m.group(3)) if m.group(3) else None}

def connector_between(prev, curr):
    if prev is None or curr is None or prev["condition"] != curr["condition"]:
        return None
    return "OR" if prev["group"] == curr["group"] else "AND"


# ══════════════════════════════════════════════════════════
# 좌표 유틸
# ══════════════════════════════════════════════════════════
def col_of(w):
    if w["x0"] < COL1_MAX: return "COND"
    if w["x0"] < COL2_MAX: return "ACT"
    return "TIME"

def join_words(ws):
    ws = sorted(ws, key=lambda w: (round(w["top"]), w["x0"]))
    return " ".join(w["text"] for w in ws)

def build_bands(anchor_tops, bottom):
    tops = sorted(set(anchor_tops))
    return [(tops[i], tops[i + 1] if i + 1 < len(tops) else bottom)
            for i in range(len(tops))]

def words_in_band(ws, top, bottom):
    return [w for w in ws if top <= w["top"] < bottom]


# ══════════════════════════════════════════════════════════
# 영역 분할 + LCO id 추출
# ══════════════════════════════════════════════════════════
def find_col_header(words):
    anchor = [w for w in words if w["text"] == "COMPLETION"]
    if not anchor:
        return None
    top = min(w["top"] for w in anchor)
    row = [w for w in words if abs(w["top"] - top) < 6]
    return (min(w["top"] for w in row), max(w["bottom"] for w in row))

def extract_lco_and_title(words):
    head = [w for w in words if w["top"] < HEADER_MARGIN]
    head_txt = join_words(head)
    ids = [x for x in RE_LCO_HEADER.findall(head_txt) if x.count('.') == 2]
    lco = ids[0] if ids else None
    title = None
    if lco:
        # LCO 번호 앞 텍스트 = 제목 (러닝헤더 좌측)
        m = re.search(rf'(.*?)\s*{re.escape(lco)}', head_txt)
        if m and m.group(1).strip():
            title = m.group(1).strip()
    return lco, title

def page_regions(words, page):
    head = [w for w in words if w["top"] < HEADER_MARGIN]
    hdr_bot = max((w["bottom"] for w in head), default=0.0)
    foot_tops = [w["top"] for w in words
                 if RE_FOOTER_LEFT.search(w["text"]) or RE_FOOTER_LCO.match(w["text"])
                 or RE_FOOTER_REV.match(w["text"])]
    foot_top = min(foot_tops) if foot_tops else page.height
    ch = find_col_header(words)
    if ch:
        return {"narr": (hdr_bot, ch[0]), "tbl": (ch[1], foot_top)}
    return {"narr": (hdr_bot, foot_top), "tbl": None}


# ══════════════════════════════════════════════════════════
# 페이지 파싱 → raw 밴드 (조건 / 액션·연결자), LCO 태깅
# ══════════════════════════════════════════════════════════
def parse_page_raw(page):
    words = page.extract_words()
    if not words:                      # 빈 페이지
        return None
    lco, title = extract_lco_and_title(words)
    reg = page_regions(words, page)

    # 내러티브 (LCO statement / applicability)
    n_top, n_bot = reg["narr"]
    nws = [w for w in words if n_top <= w["top"] < n_bot]
    lco_stmt, applic = None, None
    if nws:
        ntxt = RE_ACTIONS_HDR.split(clean_text(join_words(nws)))[0].strip()
        ap = RE_APPLIC.search(ntxt)
        if ap:
            stmt_raw = ntxt[:ap.start()].strip()
            applic = re.sub(r'^APPLICABILITY:?\s*', '',
                            ntxt[ap.start():].strip(), flags=re.IGNORECASE)
        else:
            stmt_raw = ntxt
            applic = None
        # 절 제목("3.4.15 Title")과 LCO 본문("LCO 3.4.15 The following...") 분리
        m = re.search(r'(LCO\s+\d+\.\d+\.\d+\s+.*)', stmt_raw, re.DOTALL)
        lco_stmt = m.group(1).strip() if m else (stmt_raw or None)

    # 표 (조건 밴드 / 액션 밴드)
    cond_bands_raw, act_items_raw = [], []
    if reg["tbl"]:
        t_top, t_bot = reg["tbl"]
        tws = [w for w in words if t_top <= w["top"] < t_bot]
        cond_ws = [w for w in tws if col_of(w) == "COND"]
        act_ws  = [w for w in tws if col_of(w) == "ACT"]
        time_ws = [w for w in tws if col_of(w) == "TIME"]

        act_sorted = sorted(act_ws, key=lambda w: (round(w["top"]), w["x0"]))
        act_anchors = []
        for i, w in enumerate(act_sorted):
            t = w["text"]
            if RE_ACTION_LABEL.match(t) or RE_CONNECTOR.match(t):
                act_anchors.append(w)
            elif t == '[' and i + 1 < len(act_sorted) \
                 and RE_ACTION_LABEL.match(act_sorted[i + 1]["text"]):
                act_anchors.append(act_sorted[i + 1])
        cond_anchors = [w for w in cond_ws if RE_COND_LABEL.match(w["text"])]

        for top, bottom in build_bands([w["top"] for w in cond_anchors], t_bot) if cond_anchors else []:
            raw = join_words(words_in_band(cond_ws, top, bottom))
            m = re.match(r'^([A-Z])\.', raw)
            if m:
                cond_bands_raw.append({
                    "letter": m.group(1),
                    "text": RE_STRIP_CLABEL.sub('', clean_text(raw)),
                    "optional": is_optional(raw),
                    "compound": bool(re.search(r'\bAND\b|\bOR\b', clean_text(raw))),
                })

        for top, bottom in build_bands([w["top"] for w in act_anchors], t_bot) if act_anchors else []:
            bw = sorted(words_in_band(act_ws, top, bottom), key=lambda w: (round(w["top"]), w["x0"]))
            if not bw:
                continue
            first = None
            for w in bw:
                cand = strip_bracket(w["text"])
                if cand:  # [ 단독이면 '' → skip
                    first = cand
                    break
            if first is None:
                continue
            raw_act = join_words(words_in_band(act_ws, top, bottom))
            raw_ct  = join_words(words_in_band(time_ws, top, bottom))
            if RE_CONNECTOR.match(first):
                act_items_raw.append({"type": "connector", "text": first})
            elif RE_ACTION_LABEL.match(first):
                cleaned = clean_text(raw_act)
                body = RE_STRIP_ALABEL.sub('', cleaned)
                act_items_raw.append({
                    "type": "action", "label": first,
                    "note": extract_note(body),
                    "text": strip_note(body),
                    "ct": clean_text(raw_ct),
                    "optional": is_optional(raw_act),
                    "refs": RE_SR_REF.findall(cleaned),
                })
    return {"lco": lco, "title": title, "lco_stmt": lco_stmt,
            "applicability": applic, "cond_bands": cond_bands_raw,
            "act_items": act_items_raw}


# ══════════════════════════════════════════════════════════
# 섹션 병합 (같은 LCO의 연속 페이지 → 1 섹션) + connector 계산
# ══════════════════════════════════════════════════════════
def assemble_sections(pages_data):
    sections = OrderedDict()   # lco -> accumulator
    order = []
    for pd in pages_data:
        lco = pd["lco"]
        if lco is None:
            continue
        if lco not in sections:
            sections[lco] = {"lco": lco, "title": pd["title"],
                             "lco_stmt": None, "applicability": None,
                             "cond_bands": [], "act_items": [], "pages": []}
            order.append(lco)
        s = sections[lco]
        s["pages"].append(pd["page_no"])
        if pd["lco_stmt"] and not s["lco_stmt"]:
            s["lco_stmt"] = pd["lco_stmt"]
        if pd["applicability"] and not s["applicability"]:
            s["applicability"] = pd["applicability"]
        if pd["title"] and not s["title"]:
            s["title"] = pd["title"]
        s["cond_bands"].extend(pd["cond_bands"])
        s["act_items"].extend(pd["act_items"])
    return [sections[l] for l in order]


def build_records(section):
    """섹션 → hierarchical 레코드 + actions_text + flat 레코드들"""
    lco = section["lco"]
    cond_map = {c["letter"]: c for c in section["cond_bands"]}

    # 액션에 조건/그룹/연결자 부여 (섹션 전체 순회, prev는 섹션 단위)
    prev, actions_seq = None, []
    pending_conn = None                      # 직전에 나온 원문 connector
    for it in section["act_items"]:
        if it["type"] == "connector":
            pending_conn = it["text"]        # AND / OR 저장
            actions_seq.append({"type": "connector", "text": it["text"]})
            continue
        info = parse_label(it["label"])
        actions_seq.append({"type": "action", **it, **info,
                            "connector": pending_conn})   # ← 원문 connector 사용
        pending_conn = None                  # 소비
        prev = info

    # 조건별 그룹핑 (연결자는 직전 액션 조건에 귀속)
    grouped = OrderedDict()
    cur = None
    for it in actions_seq:
        if it["type"] == "action":
            cur = it["condition"]
        if cur is not None:
            grouped.setdefault(cur, []).append(it)

    # ── actions_text (AND/OR 포함, 섹션 통합) + text_offset ──
    buf = ""
    offsets = {}   # global_id -> [start, end]
    def emit(line, gid=None):
        nonlocal buf
        start = len(buf); buf += line; end = len(buf); buf += "\n"
        if gid:
            offsets[gid] = [start, end]

    if section["title"]:
        emit(section["title"], f"{lco}/TITLE")   # "RCS Leakage Detection Instrumentation"

    if section["lco_stmt"]:
        emit(section["lco_stmt"])          # ← a/b/c 포함 본문
    if section["applicability"]:
        emit(f"APPLICABILITY: {section['applicability']}")
    buf += "\n"
    for letter, items in grouped.items():
        c = cond_map.get(letter, {"text": "", "optional": False})
        emit(f"CONDITION {letter}: {c['text']}", f"{lco}/{letter}")
        for it in items:
            if it["type"] == "connector":
                emit(f"  {it['text']}")
            else:
                note = f"(NOTE: {it['note']}) " if it["note"] else ""
                ct = f"  [CT] {it['ct']}" if it["ct"] else ""
                emit(f"  {it['label']} {note}{it['text']}{ct}",
                     f"{lco}/{it['label']}")
        buf += "\n"
    actions_text = buf.rstrip()

    # ── hierarchical 레코드 ──
    condition_blocks = []
    for ci, (letter, items) in enumerate(grouped.items(), 1):
        c = cond_map.get(letter, {"text": "", "optional": False, "compound": False})
        acts = []
        for ai, it in enumerate([x for x in items if x["type"] == "action"], 1):
            acts.append({
                "id": ai, "gid": f"{lco}/{it['label']}", "label": it["label"],
                "group": it["group"], "alt": it["alt"], "connector": it["connector"],
                "optional": it["optional"], "text": it["text"],
                "completion_time": it["ct"] or None,
                "note": it["note"], "refs": it["refs"],
            })
        condition_blocks.append({
            "id": ci, "gid": f"{lco}/{letter}", "label": letter,
            "text": c["text"], "optional": c["optional"], "compound": c.get("compound", False),
            "actions": acts,
        })

    hier = {
        "id": lco,
        "metadata": {
            "lco": lco, "title": section["title"],
            "applicability": section["applicability"],
            "source_doc": SOURCE_DOC, "source_pages": section["pages"],
        },
        "content": {
            "lco_statement": section["lco_stmt"],
            "actions_text": actions_text,      # ← naive/sliding/semantic 공통 입력
            "condition_blocks": condition_blocks,
        },
    }

    # ── flat 레코드 (condition-action 단위) ──
    flats = []
    for cb in condition_blocks:
        for a in cb["actions"]:
            ref_str = (" " + " ".join(f"See {r}." for r in a["refs"])) if a["refs"] else ""
            note_str = f" Note: {a['note']}." if a["note"] else ""
            body = (f"LCO {lco} {section['title'] or ''}. "
                    f"Applicability: {section['applicability']}. "
                    f"Condition {cb['label']}: {cb['text']} "
                    f"Required Action {a['label']}: {a['text']}"
                    f"{note_str}"
                    f" Completion Time: {a['completion_time']}.{ref_str}").strip()
            flats.append({
                "id": a["gid"],
                "metadata": {
                    "source_hierarchical_id": lco,
                    "lco": lco, "title": section["title"],
                    "applicability": section["applicability"],
                    "condition_id": cb["id"], "condition_label": cb["label"],
                    "action_id": a["id"], "action_label": a["label"],
                    "connector": a["connector"], "optional": a["optional"],
                    "source_doc": SOURCE_DOC, "source_pages": section["pages"],
                },
                "content": {
                    "condition_text": cb["text"], "action_text": a["text"],
                    "completion_time": a["completion_time"],
                    "note": a["note"], "refs": a["refs"], "body": body,
                },
            })
    if section["lco_stmt"]:
        flats.append({
            "id": f"{lco}/LCO",
            "metadata": {
                "source_hierarchical_id": lco, "lco": lco, "title": section["title"],
                "applicability": section["applicability"],
                "condition_id": None, "condition_label": None,
                "action_id": None, "action_label": None,
                "chunk_type": "lco_statement",
                "source_doc": SOURCE_DOC, "source_pages": section["pages"],
            },
            "content": {
                "condition_text": None, "action_text": None,
                "completion_time": None, "note": None, "refs": [],
                "body": (f"LCO {lco} {section['title'] or ''}. "
                        f"Applicability: {section['applicability']}. "
                        f"{section['lco_stmt']}").strip(),
            },
        })

    return hier, flats

# ══════════════════════════════════════════════════════════
# 실행: 페이지 범위 추출 → 빈 페이지 제거 → 섹션화 → 저장
# ══════════════════════════════════════════════════════════
import os
os.makedirs(OUT_DIR, exist_ok=True)

pages_data, skipped = [], []
with pdfplumber.open(PDF_PATH) as pdf:
    words = pdf.pages[244].extract_words()  # 245p
    for w in words:
        if 'D.2' in w['text'] or 'E.2' in w['text'] or '[' in w['text']:
            print(repr(w['text']), round(w['x0']))
    for pno in range(START_PAGE, END_PAGE + 1):
        page = pdf.pages[pno - 1]
        pd = parse_page_raw(page)
        if pd is None:                       # 빈 페이지
            skipped.append(pno)
            continue
        pd["page_no"] = pno
        pages_data.append(pd)

sections = assemble_sections(pages_data)

hier_records, flat_records = [], []
for s in sections:
    h, f = build_records(s)
    hier_records.append(h)
    flat_records.extend(f)

with open(f"{OUT_DIR}/hierarchical.jsonl", "w", encoding="utf-8") as fh:
    for r in hier_records:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
with open(f"{OUT_DIR}/flat.jsonl", "w", encoding="utf-8") as fh:
    for r in flat_records:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")

# ── 요약 로그 ──
print(f"페이지 범위: {START_PAGE}~{END_PAGE}")
print(f"빈 페이지 제거: {skipped or '없음'}")
print(f"섹션 수: {len(hier_records)}  |  flat chunk 수: {len(flat_records)}")
for h in hier_records:
    letters = [cb["label"] for cb in h["content"]["condition_blocks"]]
    # letter 연속성 검증
    expected = [chr(ord('A') + i) for i in range(len(letters))]
    warn = "" if letters == expected else f"  ⚠ letter 불연속 {letters}"
    print(f"  [{h['id']}] {h['metadata']['title']}  "
          f"pages={h['metadata']['source_pages']}  conditions={letters}{warn}")

print("\n── 샘플: 첫 섹션 actions_text ──")
print(hier_records[0]["content"]["actions_text"] if hier_records else "(없음)")
for r in flat_records:
    print(r["metadata"].get("action_label"), r["id"])
for h in hier_records:
    for cb in h["content"]["condition_blocks"]:
        for a in cb["actions"]:
            print(cb["label"], a["label"], "grp", a["group"], "conn", a["connector"])

'D.2.1' 234
'[' 234
'D.2.2' 240
'[' 98
'[' 98
'[' 98
'[' 234
'E.2' 240
페이지 범위: 243~247
빈 페이지 제거: 없음
섹션 수: 1  |  flat chunk 수: 17
  [3.4.15] RCS Leakage Detection Instrumentation  pages=[243, 244, 245, 246, 247]  conditions=['A', 'B', 'C', 'D', 'E', 'F', 'G']

── 샘플: 첫 섹션 actions_text ──
RCS Leakage Detection Instrumentation
LCO 3.4.15 The following RCS leakage detection instrumentation shall be OPERABLE: a. One containment sump (level or discharge flow) monitor, b. One containment atmosphere radioactivity monitor (gaseous or particulate), and c. One containment air cooler condensate flow rate monitor.
APPLICABILITY: MODES 1, 2, 3, and 4.

CONDITION A: Required containment sump monitor inoperable.
  A.1 (NOTE: Not required until 12 hours after establishment of steady state operation.) Perform SR 3.4.13.1.  [CT] Once per 24 hours
  AND
  A.2 Restore required containment sump monitor to OPERABLE status.  [CT] 30 days

CONDITION B: Required containment atmosphere radioactivity monitor inop